In [1]:

import os
import logging, timeit
#from btEngine2.DataLoader import DataLoader
from btEngine2.MarketData import MarketData
from btEngine2.TradingRule import TradingRule

import polars as pl
import numpy as np  


import platform
import pandas as pd


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.options.display.float_format = lambda x: f'{x:,.0f}' if abs(x) >= 1000 else (f'{x:.2f}' if abs(x) < 10 else f'{x:.1f}')
pd.set_option('future.no_silent_downcasting', True)

pd.set_option('display.max_rows', 40)

# Detect operating system
if platform.system() == "Windows":
    ticker_csv_path = r'G:\Projects\BackTesting1.0\Data\Inputs\TickerList-Futs.csv'
    save_directory = r"G:\Projects\BackTesting1.0\Data\Bloomberg\Futures"
    helper_directory = r'G:\Projects\BackTesting1.0\Data\Bloomberg\HelperFiles'
    bt_folder = r'BackTests\seasn_research'
    av_folder = r'G:\Projects\BackTesting1.0\Data\Inputs\AssetSizing-Futs.csv'
else:  # Assume macOS for other cases
    ticker_csv_path = r'Data/Inputs/TickerList-Futs.csv'
    save_directory = r"Data/Bloomberg/Futures"
    helper_directory = r'Data/Bloomberg/HelperFiles'
    bt_folder = r'BackTests/seasn_research'
    av_folder = r'Data/Inputs/AssetSizing-Futs.csv'


# Define paths to auxiliary data for MarketData
tick_values_path = os.path.join(helper_directory, 'fut_val_pt.parquet')
fx_rates_path = os.path.join(helper_directory, 'fxHist.parquet')

# Initialize the MarketData
market_data = MarketData(
    base_directory=save_directory,
    tick_values_path=tick_values_path,
    fx_rates_path=fx_rates_path,
    instrument_type="Futures",
    n_threads=8,  # Number of threads for parallel data loading
    log_level=logging.ERROR  # Set to DEBUG for more detailed logs
)



In [2]:
tick = 'BTC1 Curncy'
# Access data for a specific ticker
try:
    test_df = market_data.get_ticker_data(tick)
    print(test_df)
except ValueError as e:
    print(e)

# Access all preprocessed data
all_data = market_data.get_data()
print(f"Total tickers loaded: {len(all_data)}")

# Access FX rates
fx_rates = market_data.get_fx_rates()
# Access tick values
tick_values = market_data.get_tick_values()
# Access asset classes
asset_classes = market_data.get_asset_classes()

#market_data = market_data.date_filter(start_date='01012010')

shape: (1_759, 14)
┌────────────┬──────────┬──────────┬─────────┬───┬─────────┬─────────┬──────────────┬──────────────┐
│ Date       ┆ Open     ┆ High     ┆ Low     ┆ … ┆ BadOHLC ┆ FX_Rate ┆ Tick_Value_B ┆ Tick_Value_U │
│ ---        ┆ ---      ┆ ---      ┆ ---     ┆   ┆ ---     ┆ ---     ┆ ase          ┆ SD           │
│ date       ┆ f64      ┆ f64      ┆ f64     ┆   ┆ bool    ┆ f64     ┆ ---          ┆ ---          │
│            ┆          ┆          ┆         ┆   ┆         ┆         ┆ f64          ┆ f64          │
╞════════════╪══════════╪══════════╪═════════╪═══╪═════════╪═════════╪══════════════╪══════════════╡
│ 2017-12-18 ┆ 35030.0  ┆ 35030.0  ┆ 32725.0 ┆ … ┆ false   ┆ 1.0     ┆ 5.0          ┆ 5.0          │
│ 2017-12-19 ┆ 33515.0  ┆ 34105.0  ┆ 31560.0 ┆ … ┆ false   ┆ 1.0     ┆ 5.0          ┆ 5.0          │
│ 2017-12-20 ┆ 32125.0  ┆ 32730.0  ┆ 30815.0 ┆ … ┆ false   ┆ 1.0     ┆ 5.0          ┆ 5.0          │
│ 2017-12-21 ┆ 30780.0  ┆ 31650.0  ┆ 29460.0 ┆ … ┆ false   ┆ 1.0     ┆ 5

In [4]:
from btEngine2.Rules.Seasonality.weekly_seasn import *

pSizeParamsUse = {
    'AssetVol': 1000000,  # Target asset volatility in USD
    'VolLookBack': 30,
    'VolMethod': 'ewm'  # Lookback period for volatility calculation
}

In [ ]:

    
hr_t_long = [0.53, 0.75]
hr_t_short = [0.47, 0.25]
ret_t = [1.75, 1.25]
risk_scale = [1, 2]


params_b_rescale ={
    'X_years': 6,
    'ewma_span': 6,
    'hr_threshold_long': hr_t_long,
    'hr_threshold_short': hr_t_short,
    'ret_threshold': ret_t,
    'min_lb': 5,
    'wks_to_trade': [], # [2,3,6,9,10,11,15,16,22,27,32,34,36, 37,40, 41,42,44,45,46,47,48,52],
    'trade_direction': 'both',
    'risk_scale': risk_scale,
    'add_days': 30
}

wklySeasn_b_rescale = TradingRule(
    market_data=market_data,
    trading_rule_function=wkly_seasn_simple,
    trading_params=params_b_rescale,
    position_sizing_params=pSizeParamsUse,
    cont_rule=False,
    bt_folder=bt_folder,
    excl_assets=['comm-soft'],
    strat_descr='Seasonality Strategy Long/Short with TF (21 EMA) and Vol Filter (15, 45, 0.75), Hi+Lo HR (1:2)'
    
)